# capture: square crops + automatic labels from a PGN replay (M2)

Data collection without hand labeling: replay a recorded game (PGN) on the physical board,
one position per ply. python-chess knows the true board at every ply, so each warped frame
is labeled automatically: 64 crops, 64 labels, 64 manifest rows. Output format is the
contract in `docs/contracts.md` (read back by `dataset.ipynb`):

```
data/crops/<session_id>/frames/<frame_id>.png
                        squares/<frame_id>_<square>.png
                        manifest.csv
```

Crop geometry (`configs/camera.yaml: crop`): the square, widened by `widen` squares on the
left and on the right, extended by `extend_up` squares toward the top of the warped image
(pieces are tall and lean away from the camera, i.e. upward in the `black_bottom` warp),
then resized to `size_wh`. Parts outside the warp are padded by edge replication.
Note: with the defaults (widen 0.5, extend_up 1.0) the region is 2 x 2 squares and is resized
to 100 x 200, i.e. squeezed 2x horizontally. That is the contract as written; revisit
(e.g. widen 0.0 -> 1 x 2 squares, no distortion) before collecting a lot of data.

Milestone: M2 (data collection). Run with `run_capture.ipynb`.

**TODO**
- [ ] `capture_game`: live camera loop (prompt the human to set each position, grab, undistort,
      warp, crop, write)
- [ ] decide the final crop numbers (see the note above) and record them in camera.yaml
- [ ] simulated sessions: same writer fed by Gazebo frames (`gambit_sim` data generator)

In [ ]:
from __future__ import annotations

import csv
import io
from pathlib import Path

import chess
import chess.pgn
import numpy as np

%run ./homography.ipynb
%run ./dataset.ipynb

## Crop geometry

In [ ]:
def crop_box(sq: int, geom: BoardGeometry, warp, crop) -> tuple[int, int, int, int]:
    """(x0, y0, x1, y1) of square sq's crop in the warp image (may extend past the edges)."""
    x0, y0, x1, y1 = square_cell_warp(sq, geom, warp)
    w, h = x1 - x0, y1 - y0
    return (int(round(x0 - crop.widen * w)), int(round(y0 - crop.extend_up * h)),
            int(round(x1 + crop.widen * w)), int(y1))


def extract_crop(warped: np.ndarray, sq: int, geom: BoardGeometry, warp, crop) -> np.ndarray:
    """One crop, resized to crop.size_wh; out-of-image parts are edge-replicated."""
    import cv2
    x0, y0, x1, y1 = crop_box(sq, geom, warp, crop)
    H, W = warped.shape[:2]
    pad = max(0, -x0, -y0, x1 - W, y1 - H)
    img = warped
    if pad:
        img = cv2.copyMakeBorder(warped, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
    patch = img[y0 + pad:y1 + pad, x0 + pad:x1 + pad]
    return cv2.resize(patch, tuple(int(v) for v in crop.size_wh), interpolation=cv2.INTER_AREA)


def extract_all_crops(warped: np.ndarray, geom: BoardGeometry, warp, crop) -> list[np.ndarray]:
    """64 crops in square-index order (a1..h8)."""
    return [extract_crop(warped, sq, geom, warp, crop) for sq in chess.SQUARES]

## Auto-labeling from a PGN
`pgn_positions` yields (move_idx, board) for the start position and after every ply of the
mainline; `label_rows` turns one board into the 64 manifest rows.

In [ ]:
def pgn_positions(pgn: str | Path) -> list[tuple[int, chess.Board]]:
    """[(0, start), (1, after ply 1), ...] for the first game in a PGN file or PGN text."""
    text = Path(pgn).read_text() if (isinstance(pgn, Path) or (isinstance(pgn, str) and pgn.endswith(".pgn"))) else pgn
    game = chess.pgn.read_game(io.StringIO(text))
    if game is None:
        raise ValueError("no game found in PGN")
    board = game.board()
    out = [(0, board.copy())]
    for i, move in enumerate(game.mainline_moves(), start=1):
        board.push(move)
        out.append((i, board.copy()))
    return out


def label_rows(session_id: str, game_id: str, move_idx: int, frame_id: str,
               board: chess.Board) -> list[dict]:
    """64 manifest rows (MANIFEST_COLUMNS) for one frame, square-index order."""
    labels = board_to_labels(board)
    fen = board.fen()
    return [{"session_id": session_id, "game_id": game_id, "move_idx": move_idx,
             "frame_id": frame_id, "square": SQUARE_NAMES[sq], "label": labels[sq],
             "path": f"squares/{frame_id}_{SQUARE_NAMES[sq]}.png", "fen": fen}
            for sq in chess.SQUARES]

## Writing a session
`ManifestWriter` appends rows to `manifest.csv` (header written once) and hands out
frame ids (`000000`, `000001`, ... continuing after existing frames).

In [ ]:
class ManifestWriter:
    def __init__(self, crops_root: str | Path, session_id: str):
        self.session_id = session_id
        self.dir = Path(crops_root) / session_id
        (self.dir / "squares").mkdir(parents=True, exist_ok=True)
        (self.dir / "frames").mkdir(parents=True, exist_ok=True)
        self.manifest = self.dir / "manifest.csv"
        existing = sorted((self.dir / "frames").glob("*.png"))
        self._next = int(existing[-1].stem) + 1 if existing else 0

    def next_frame_id(self) -> str:
        fid = f"{self._next:06d}"
        self._next += 1
        return fid

    def append(self, rows: list[dict]) -> None:
        new = not self.manifest.exists()
        with open(self.manifest, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(MANIFEST_COLUMNS))
            if new:
                w.writeheader()
            w.writerows(rows)

    def write_frame(self, warped: np.ndarray, crops: list[np.ndarray], game_id: str,
                    move_idx: int, board: chess.Board) -> str:
        """Save the warped frame, its 64 crops (BGR, as from cv2) and manifest rows."""
        import cv2
        if len(crops) != 64:
            raise ValueError(f"expected 64 crops, got {len(crops)}")
        fid = self.next_frame_id()
        cv2.imwrite(str(self.dir / "frames" / f"{fid}.png"), warped)
        rows = label_rows(self.session_id, game_id, move_idx, fid, board)
        for row, c in zip(rows, crops):
            cv2.imwrite(str(self.dir / row["path"]), c)
        self.append(rows)
        return fid

## Live capture loop (TODO)

In [ ]:
def capture_game(camera, pgn: str | Path, writer: ManifestWriter, game_id: str,
                 geom: BoardGeometry, cam_cfg, intrinsics: dict | None = None,
                 frames_per_position: int = 1) -> int:
    """TODO M2: replay one game on the physical board and record it.

    For each (move_idx, board) in pgn_positions(pgn): print the move to make (or the full
    position), wait for the human (keypress), grab `frames_per_position` frames (vary hand
    placement / lighting between them if you like), undistort with `intrinsics`
    (calibration.ipynb: undistort), compute_homography + warp_board, extract_all_crops, then
    writer.write_frame(warped, crops, game_id, move_idx, board). Skip frames where the
    markers are not found. Show draw_grid_overlay as a live sanity check.
    Returns the number of frames written.
    """
    raise NotImplementedError("TODO M2: capture_game")

## Self-check
Crop geometry on the default config, crops from a synthetic warped board, PGN labeling and a
manifest round trip through `dataset.ipynb: load_manifest`.

In [ ]:
import tempfile
from types import SimpleNamespace

_cp_cam = load_config("configs/camera.yaml")
_cp_geom = load_board_geometry()
_w, _c = _cp_cam.warp, _cp_cam.crop
# a1 cell is (800, 100, 900, 200) in the black_bottom warp -> crop 2 squares wide, 2 tall
assert square_cell_warp(chess.A1, _cp_geom, _w) == (800, 100, 900, 200)
assert crop_box(chess.A1, _cp_geom, _w, _c) == (750, 0, 950, 200)
assert crop_box(chess.H8, _cp_geom, _w, _c) == (50, 700, 250, 900)

_warped = render_synthetic_board(_cp_geom, SimpleNamespace(square_px=100, margin_px=150, view="black_bottom"))
_warped = _warped[50:-50, 50:-50]   # same size as the default warp (1000 x 1000)
_crops = extract_all_crops(_warped, _cp_geom, _w, _c)
assert len(_crops) == 64 and all(c.shape == (200, 100, 3) for c in _crops)

_pos = pgn_positions('[Event "t"]\n\n1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 4. O-O *')
assert len(_pos) == 8 and _pos[-1][1].piece_at(chess.G1) == chess.Piece.from_symbol("K")
_rows = label_rows("s_test", "g1", 1, "000001", _pos[1][1])
assert len(_rows) == 64 and _rows[chess.E4]["label"] == "P" and _rows[chess.E2]["label"] == "empty"
assert _rows[0]["path"] == "squares/000001_a1.png"

with tempfile.TemporaryDirectory() as _t:
    _wr = ManifestWriter(_t, "s_test")
    for _mi, _b in _pos[:3]:
        _wr.write_frame(_warped, _crops, "g1", _mi, _b)
    _df = load_manifest(_t, ["s_test"])
    assert len(_df) == 3 * 64 and list(_df.frame_id.unique()) == ["000000", "000001", "000002"]
    assert Path(_df.abs_path.iloc[0]).exists()
    assert ManifestWriter(_t, "s_test").next_frame_id() == "000003"
print("capture OK")